In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Classroom Demo")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.cores.max", "4")
    .getOrCreate()
)

print(spark.version)

4.3.0


In [10]:
events_df = spark.read.json(
    "/opt/spark-data/input/events.json"
)

events_df.printSchema()

root
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- event_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- page: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- user_id: string (nullable = true)



In [11]:
events_df.show(10)

+-------+-------+--------+-------------+-------------------+----------+----------+-------------------+-------+
|country| device|event_id|   event_type|               page|product_id|session_id|          timestamp|user_id|
+-------+-------+--------+-------------+-------------------+----------+----------+-------------------+-------+
|  Spain|desktop|       1|    page_view|                  /|      NULL|     S1001|2026-11-19T09:00:00|   U001|
|  Spain|desktop|       2|    page_view|          /products|      NULL|     S1001|2026-11-19T09:01:12|   U001|
|  Spain|desktop|       3|product_click|/products/laptop-01|      P001|     S1001|2026-11-19T09:02:30|   U001|
|  Spain|desktop|       4|  add_to_cart|              /cart|      P001|     S1001|2026-11-19T09:03:15|   U001|
|  Spain|desktop|       5|     purchase|  /checkout/success|      P001|     S1001|2026-11-19T09:05:42|   U001|
| France| mobile|       6|    page_view|                  /|      NULL|     S1002|2026-11-19T09:10:00|   U002|
|

In [12]:
events_df.count()

15

In [13]:
events_min_df = events_df.select("country", "device")

In [14]:
events_min_df.show()

+-------+-------+
|country| device|
+-------+-------+
|  Spain|desktop|
|  Spain|desktop|
|  Spain|desktop|
|  Spain|desktop|
|  Spain|desktop|
| France| mobile|
| France| mobile|
| France| mobile|
|  Spain|desktop|
|  Spain|desktop|
|  Spain|desktop|
|Germany| mobile|
|Germany| mobile|
|Germany| mobile|
|  Spain| tablet|
+-------+-------+



In [18]:
events_df.select("country").distinct().show()

+-------+
|country|
+-------+
|Germany|
| France|
|  Spain|
+-------+



In [19]:
events_df.filter(events_df.country == "Spain").show()

+-------+-------+--------+-------------+-------------------+----------+----------+-------------------+-------+
|country| device|event_id|   event_type|               page|product_id|session_id|          timestamp|user_id|
+-------+-------+--------+-------------+-------------------+----------+----------+-------------------+-------+
|  Spain|desktop|       1|    page_view|                  /|      NULL|     S1001|2026-11-19T09:00:00|   U001|
|  Spain|desktop|       2|    page_view|          /products|      NULL|     S1001|2026-11-19T09:01:12|   U001|
|  Spain|desktop|       3|product_click|/products/laptop-01|      P001|     S1001|2026-11-19T09:02:30|   U001|
|  Spain|desktop|       4|  add_to_cart|              /cart|      P001|     S1001|2026-11-19T09:03:15|   U001|
|  Spain|desktop|       5|     purchase|  /checkout/success|      P001|     S1001|2026-11-19T09:05:42|   U001|
|  Spain|desktop|       9|    page_view|                  /|      NULL|     S1003|2026-11-19T09:15:00|   U003|
|

In [24]:
events_parquet = events_df.filter(
    (events_df.country == "Spain") &
    (events_df.event_type == "purchase")
)

In [25]:
events_parquet.write \
    .mode("overwrite") \
    .parquet("/opt/spark-data/output/events")

In [26]:
events_df.createOrReplaceTempView("events")

In [27]:
spark.sql("""
    SELECT *
    FROM events
""").show()

+-------+-------+--------+-------------+--------------------+----------+----------+-------------------+-------+
|country| device|event_id|   event_type|                page|product_id|session_id|          timestamp|user_id|
+-------+-------+--------+-------------+--------------------+----------+----------+-------------------+-------+
|  Spain|desktop|       1|    page_view|                   /|      NULL|     S1001|2026-11-19T09:00:00|   U001|
|  Spain|desktop|       2|    page_view|           /products|      NULL|     S1001|2026-11-19T09:01:12|   U001|
|  Spain|desktop|       3|product_click| /products/laptop-01|      P001|     S1001|2026-11-19T09:02:30|   U001|
|  Spain|desktop|       4|  add_to_cart|               /cart|      P001|     S1001|2026-11-19T09:03:15|   U001|
|  Spain|desktop|       5|     purchase|   /checkout/success|      P001|     S1001|2026-11-19T09:05:42|   U001|
| France| mobile|       6|    page_view|                   /|      NULL|     S1002|2026-11-19T09:10:00| 

In [28]:
purchases_df = spark.sql("""
    SELECT user_id, country, device, timestamp
    FROM events
    WHERE event_type = 'purchase'
""")

In [29]:
purchases_df.show(2)

+-------+-------+-------+-------------------+
|user_id|country| device|          timestamp|
+-------+-------+-------+-------------------+
|   U001|  Spain|desktop|2026-11-19T09:05:42|
|   U004|Germany| mobile|2026-11-19T09:23:50|
+-------+-------+-------+-------------------+



In [30]:
spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         |   events|       true|
+---------+---------+-----------+



In [31]:
spark.sql("DESCRIBE events").show()

+----------+---------+-------+
|  col_name|data_type|comment|
+----------+---------+-------+
|   country|   string|   NULL|
|    device|   string|   NULL|
|  event_id|   bigint|   NULL|
|event_type|   string|   NULL|
|      page|   string|   NULL|
|product_id|   string|   NULL|
|session_id|   string|   NULL|
| timestamp|   string|   NULL|
|   user_id|   string|   NULL|
+----------+---------+-------+



In [33]:
# List tables and views
spark.catalog.listTables()

# List columns
spark.catalog.listColumns("events")

# List databases
spark.catalog.listDatabases()

[Database(name='default', catalog='spark_catalog', description='default database', locationUri='file:/home/jovyan/work/spark-warehouse')]